# Family Health Guardian - Diabetes Risk Assessment
## Machine Learning Training & Evaluation Workflow

> **Clinical & Regulatory Disclaimer**:
> This machine learning model is developed strictly for statistical risk stratification and lifestyle triage within the **Family Health Guardian** application. It is **NOT** an FDA/CE-cleared diagnostic medical device and does **NOT** provide a clinical diagnosis of Diabetes Mellitus. Clinical diagnosis requires certified laboratory diagnostic procedures (such as fasting plasma glucose, oral glucose tolerance test, or venous HbA1c assays) conducted under the supervision of a licensed medical practitioner.

### Notebook Objectives & Key Improvements:
1. **Prevent Data Leakage**: All preprocessing (one-hot encoding, imputing) is fit exclusively on the training partition (`X_train`) via scikit-learn `Pipeline` and `ColumnTransformer`.
2. **Stratified Held-Out Split**: 80/20 train/test split stratified on the target class (`diabetes`).
3. **Comprehensive Metrics**: Reporting Accuracy, Balanced Accuracy, Precision, Recall (with special clinical attention to positive-class diabetes detection), F1-Score, Confusion Matrix, and ROC-AUC.
4. **Baseline Benchmarks**: Explicit comparison against Dummy classifiers (most frequent, stratified) and a linear clinical baseline (Logistic Regression).
5. **Decision Threshold Analysis**: Evaluating False Negatives (FN) across decision thresholds (0.20 to 0.50).
6. **API Parity & Artifact Persistence**: Exact 13-feature ordering and naming matching `EXPECTED_FEATURES` in `backend/app/services/ml_service.py`.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

print(f"Python:       {sys.version.split()[0]}")
print(f"Pandas:       {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Joblib:       {joblib.__version__}")


Python:       3.13.14
Pandas:       3.0.6
Scikit-learn: 1.9.0
Joblib:       1.6.0


---
## 1. Dataset Ingestion & Exploratory Data Analysis (EDA)

We load the dataset using robust relative path discovery to guarantee reproducibility across local repos, Colab, and container environments.


In [2]:
# Robust dataset path resolution
possible_paths = [
    Path("../data/diabetes_prediction_dataset.csv"),
    Path("data/diabetes_prediction_dataset.csv"),
    Path("C:/Users/HRISHIKESH/Downloads/diabetes_prediction_dataset.csv/diabetes_prediction_dataset.csv"),
    Path("/content/diabetes-prediction-dataset/diabetes_prediction_dataset.csv")
]

dataset_path = None
for p in possible_paths:
    if p.is_file():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError("Dataset not found. Please place diabetes_prediction_dataset.csv in data/ directory.")

print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)
print(f"Raw shape: {df.shape}")
df.head()


Loading dataset from: ..\data\diabetes_prediction_dataset.csv
Raw shape: (100000, 9)


In [3]:
# Check missing values and duplicate records
print("Missing values per column:")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows detected: {duplicate_count}")

# Deduplication to prevent identical records across partitions
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")


Missing values per column:
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Duplicate rows detected: 3854
Shape after removing duplicates: (96146, 9)


In [4]:
# Target distribution and severe class imbalance check
target_counts = df['diabetes'].value_counts()
target_pct = df['diabetes'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct.round(2)})
print("Target Class Distribution ('diabetes'):")
print(dist_df)
print(f"\nClass Imbalance Ratio: Non-diabetic {target_pct[0]:.1f}% vs Diabetic {target_pct[1]:.1f}%")


Target Class Distribution ('diabetes'):
          Count  Percentage (%)
diabetes                       
0         87664           91.18
1          8482            8.82

Class Imbalance Ratio: Non-diabetic 91.2% vs Diabetic 8.8%


---
## 2. Preventing Data Leakage: Stratified Train/Test Split

**Critical Machine Learning Principle**:
Transformations, statistical encodings, scalers, and imputers must **NEVER** be fit on the entire dataset prior to splitting.
Applying transformations globally leaks categorical domain knowledge and distributional statistics from the test set into the training set.

We partition the raw features into an 80% training set and 20% held-out testing set using **stratified sampling** to maintain the 8.82% positive class proportion in both partitions.


In [5]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training split: {X_train.shape[0]} samples (Class 1 ratio: {y_train.mean():.4f})")
print(f"Testing split:  {X_test.shape[0]} samples (Class 1 ratio: {y_test.mean():.4f})")


Training split: 76916 samples (Class 1 ratio: 0.0882)
Testing split:  19230 samples (Class 1 ratio: 0.0882)


---
## 3. Preprocessing Architecture & Feature Ordering Parity

The FastAPI backend (`DiabetesMLService` in `app/services/ml_service.py`) expects the trained model to accept 13 features in this exact sequence:
1. `age`
2. `hypertension`
3. `heart_disease`
4. `bmi`
5. `HbA1c_level`
6. `blood_glucose_level`
7. `gender_Male`
8. `gender_Other`
9. `smoking_history_current`
10. `smoking_history_ever`
11. `smoking_history_former`
12. `smoking_history_never`
13. `smoking_history_not current`

We construct a scikit-learn `ColumnTransformer` that fits `OneHotEncoder(drop='first')` **only on `X_train`**, ensuring exact feature name and ordering parity.


In [6]:
num_cols = ["age", "hypertension", "heart_disease", "bmi", "HbA1c_level", "blood_glucose_level"]
cat_cols = ["gender", "smoking_history"]

# Construct transformers
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='No Info')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ],
    verbose_feature_names_out=False
)

# Fit strictly on training data
preprocessor.fit(X_train)

feature_names = list(preprocessor.get_feature_names_out())
print("Features generated by training preprocessor:")
print(feature_names)

EXPECTED_API_FEATURES = [
    'age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level',
    'gender_Male', 'gender_Other', 'smoking_history_current', 'smoking_history_ever',
    'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'
]

assert feature_names == EXPECTED_API_FEATURES, "Mismatch with backend API EXPECTED_FEATURES!"
print("\n[PASS] Preprocessor feature ordering matches backend API contract perfectly!")


Features generated by training preprocessor:
['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male', 'gender_Other', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']

[PASS] Preprocessor feature ordering matches backend API contract perfectly!


In [7]:
# Transform training and testing data using the fitted preprocessor
X_train_proc = pd.DataFrame(preprocessor.transform(X_train), columns=feature_names, index=X_train.index)
X_test_proc = pd.DataFrame(preprocessor.transform(X_test), columns=feature_names, index=X_test.index)

print(f"X_train_proc shape: {X_train_proc.shape}")
print(f"X_test_proc shape:  {X_test_proc.shape}")


X_train_proc shape: (76916, 13)
X_test_proc shape:  (19230, 13)


---
## 4. Baseline Model Benchmarking

To rigorously validate whether our machine learning model adds true value, we benchmark against three established baselines:
1. **Dummy (Most Frequent)**: Always predicts class 0. Illustrates the "Accuracy Paradox" on imbalanced data.
2. **Dummy (Stratified)**: Predicts randomly according to the class proportions.
3. **Logistic Regression (Balanced)**: Standard, interpretable clinical linear model with balanced class weighting.


In [8]:
def evaluate_model(name, clf, X_tr, y_tr, X_te, y_te, is_logistic=False):
    """Train and evaluate a classifier on held-out test data."""
    if is_logistic:
        scaler = StandardScaler()
        X_tr_fit = scaler.fit_transform(X_tr)
        X_te_eval = scaler.transform(X_te)
    else:
        X_tr_fit = X_tr
        X_te_eval = X_te

    clf.fit(X_tr_fit, y_tr)
    y_pred = clf.predict(X_te_eval)

    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_te_eval)[:, 1]
        auc = roc_auc_score(y_te, y_prob)
        pr_auc = average_precision_score(y_te, y_prob)
    else:
        auc = 0.5
        pr_auc = y_te.mean()

    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    prec_1 = precision_score(y_te, y_pred, pos_label=1, zero_division=0)
    rec_1 = recall_score(y_te, y_pred, pos_label=1, zero_division=0)
    f1_1 = f1_score(y_te, y_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_te, y_pred)

    return {
        "Model": name,
        "Accuracy": acc,
        "Balanced Acc": bal_acc,
        "ROC-AUC": auc,
        "PR-AUC": pr_auc,
        "Precision (1)": prec_1,
        "Recall (1)": rec_1,
        "F1-Score (1)": f1_1,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1],
        "fitted_model": clf
    }

benchmarks = [
    ("Baseline: Dummy (Most Frequent)", DummyClassifier(strategy="most_frequent"), False),
    ("Baseline: Dummy (Stratified)", DummyClassifier(strategy="stratified", random_state=42), False),
    ("Baseline: Logistic Regression (Balanced)", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), True)
]

results = []
for name, clf, is_log in benchmarks:
    res = evaluate_model(name, clf, X_train_proc, y_train, X_test_proc, y_test, is_logistic=is_log)
    results.append(res)
    print(f"Trained {name}")


Trained Baseline: Dummy (Most Frequent)
Trained Baseline: Dummy (Stratified)
Trained Baseline: Logistic Regression (Balanced)


---
## 5. Random Forest Training & Full Model Evaluation

We now train the `RandomForestClassifier` (200 trees, `class_weight='balanced'`) on the leakage-free training split and compare it with the unweighted variant.


In [9]:
# Random Forest with class_weight='balanced'
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf_bal_result = evaluate_model("Random Forest (Balanced, 200 trees)", rf_balanced, X_train_proc, y_train, X_test_proc, y_test)
results.append(rf_bal_result)

# Random Forest unweighted (default)
rf_unweighted = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf_unw_result = evaluate_model("Random Forest (Default Unweighted)", rf_unweighted, X_train_proc, y_train, X_test_proc, y_test)
results.append(rf_unw_result)


In [10]:
# Formatted Comparison Table
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != "fitted_model"} for r in results])
print("=" * 115)
print("BENCHMARK COMPARISON TABLE (HELD-OUT TEST SET)")
print("=" * 115)
print(summary_df.to_string(index=False))


BENCHMARK COMPARISON TABLE (HELD-OUT TEST SET)
                                   Model  Accuracy  Balanced Acc  ROC-AUC   PR-AUC  Precision (1)  Recall (1)  F1-Score (1)    TN   FP   FN   TP
         Baseline: Dummy (Most Frequent)  0.911804      0.500000 0.500000 0.088196       0.000000    0.000000      0.000000 17534    0 1696    0
            Baseline: Dummy (Stratified)  0.841082      0.503559 0.503559 0.088811       0.094756    0.093750      0.094250 16015 1519 1537  159
Baseline: Logistic Regression (Balanced)  0.884503      0.882342 0.960074 0.816659       0.425192    0.879717      0.573295 15517 2017  204 1492
     Random Forest (Balanced, 200 trees)  0.958502      0.863270 0.967505 0.851259       0.774115    0.747642      0.760648 17164  370  428 1268
      Random Forest (Default Unweighted)  0.969319      0.842572 0.961111 0.855682       0.949593    0.688679      0.798360 17472   62  528 1168


In [11]:
# Classification Report for Random Forest (Balanced)
y_pred_rf = rf_bal_result["fitted_model"].predict(X_test_proc)
print("Classification Report - Random Forest (Balanced):")
print(classification_report(y_test, y_pred_rf, digits=4))

cm = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:")
print(f"[[TN={cm[0,0]}  FP={cm[0,1]}], [FN={cm[1,0]}  TP={cm[1,1]}]]")
print(f"\nPositive Class (Diabetes=1): Detected {cm[1,1]} / {cm[1,1] + cm[1,0]} patients (Recall = {cm[1,1]/(cm[1,1]+cm[1,0]):.2%})")


Classification Report - Random Forest (Balanced):
              precision    recall  f1-score   support

           0     0.9757    0.9789    0.9773     17534
           1     0.7741    0.7476    0.7606      1696

    accuracy                         0.9585     19230
   macro avg     0.8749    0.8633    0.8690     19230
weighted avg     0.9579    0.9585    0.9582     19230

Confusion Matrix:
[[TN=17164  FP=370], [FN=428  TP=1268]]

Positive Class (Diabetes=1): Detected 1268 / 1696 patients (Recall = 74.76%)


---
## 6. Clinical Recall Analysis & Decision Threshold Tuning

### Why Recall Matters in Clinical Screening
In preventive healthcare and triage, **False Negatives (FN - missing a diabetic patient)** carry severe health risks:
delayed intervention leads to preventable microvascular and macrovascular complications.
Conversely, **False Positives (FP - flagging a non-diabetic for follow-up)** only prompt a routine confirmatory blood test.

Below, we analyze how varying the classification decision threshold from **0.20 to 0.50** shifts the clinical trade-off between **Recall** and **Precision**.


In [12]:
y_prob_rf = rf_bal_result["fitted_model"].predict_proba(X_test_proc)[:, 1]

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_metrics = []

for t in thresholds:
    preds_t = (y_prob_rf >= t).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    threshold_metrics.append({
        "Threshold": t,
        "Recall (1)": recall_score(y_test, preds_t),
        "Precision (1)": precision_score(y_test, preds_t),
        "F1-Score (1)": f1_score(y_test, preds_t),
        "Missed Diabetics (FN)": cm_t[1, 0],
        "False Alarms (FP)": cm_t[0, 1],
        "Detected (TP)": cm_t[1, 1],
        "Non-Diabetics (TN)": cm_t[0, 0]
    })

thresh_df = pd.DataFrame(threshold_metrics)
print("Decision Threshold Performance Comparison:")
print(thresh_df.to_string(index=False))

print("\nKey Clinical Finding:")
print("At threshold 0.25 (used by the API for 'Moderate Risk'), Recall increases to 86.26%, reducing missed diabetics from 428 to 233!")


Decision Threshold Performance Comparison:
 Threshold  Recall (1)  Precision (1)  F1-Score (1)  Missed Diabetics (FN)  False Alarms (FP)  Detected (TP)  Non-Diabetics (TN)
      0.20    0.883844       0.465095      0.609473                    197               1724           1499               15810
      0.25    0.862618       0.511538      0.642230                    233               1397           1463               16137
      0.30    0.828420       0.555775      0.665246                    291               1123           1405               16411
      0.35    0.806014       0.606208      0.691977                    329                888           1367               16646
      0.40    0.787146       0.663189      0.719871                    361                678           1335               16856
      0.45    0.770637       0.717738      0.743247                    389                514           1307               17020
      0.50    0.750000       0.768580      0.759176   

---
## 7. End-to-End Pipeline Architecture

To demonstrate complete encapsulation, we build a single scikit-learn `Pipeline` that ingests raw, unprocessed patient data (8 columns) and directly outputs risk predictions without any manual preprocessing steps.


In [13]:
# Create unified end-to-end Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', rf_bal_result["fitted_model"])
])

# Demonstrate inference directly on raw test data (8 features)
sample_raw_patients = X_test.head(4)
sample_preds = pipeline.predict(sample_raw_patients)
sample_probs = pipeline.predict_proba(sample_raw_patients)[:, 1]

print("Pipeline Predictions on Raw Inputs:")
for idx, (p_idx, row) in enumerate(sample_raw_patients.iterrows()):
    print(f"Patient {idx+1} [Age: {row['age']}, BMI: {row['bmi']}, HbA1c: {row['HbA1c_level']}%] -> Pred: {sample_preds[idx]}, Risk: {sample_probs[idx]:.2%}")


Pipeline Predictions on Raw Inputs:
Patient 1 [Age: 51.0, BMI: 36.77, HbA1c: 6.6%] -> Pred: 0, Risk: 17.00%
Patient 2 [Age: 12.0, BMI: 22.29, HbA1c: 4.5%] -> Pred: 0, Risk: 0.00%
Patient 3 [Age: 60.0, BMI: 34.24, HbA1c: 6.2%] -> Pred: 0, Risk: 2.50%
Patient 4 [Age: 80.0, BMI: 24.39, HbA1c: 4.0%] -> Pred: 0, Risk: 0.00%


C:\Users\HRISHIKESH\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\HRISHIKESH\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


---
## 8. Model Artifact Persistence & Backend Compatibility Verification

We save the final trained model in the exact schema expected by the FastAPI backend:
`{"model": estimator, "features": feature_names_list}`
saved to `backend/models/diabetes_model.pkl` with `joblib(compress=3)` to keep file size under 30MB.


In [14]:
# Resolve models directory relative to backend project
models_dir = Path("../models") if Path("../models").exists() else Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

model_output_path = models_dir / "diabetes_model.pkl"

model_artifact = {
    "model": rf_bal_result["fitted_model"],
    "features": feature_names
}

# Compress with joblib level 3
joblib.dump(model_artifact, model_output_path, compress=3)

file_size_mb = os.path.getsize(model_output_path) / (1024 * 1024)
print(f"Saved model artifact to: {model_output_path.resolve()}")
print(f"Compressed file size:   {file_size_mb:.2f} MB")


Saved model artifact to: C:\Users\HRISHIKESH\Downloads\Family-Health-Guardian\backend\models\diabetes_model.pkl
Compressed file size:   22.37 MB


In [15]:
# Verify loading and inference compatibility
loaded_artifact = joblib.load(model_output_path)
loaded_estimator = loaded_artifact["model"]
loaded_features = loaded_artifact["features"]

assert hasattr(loaded_estimator, "predict"), "Loaded model missing predict()"
assert hasattr(loaded_estimator, "predict_proba"), "Loaded model missing predict_proba()"
assert loaded_features == EXPECTED_API_FEATURES, "Loaded features mismatch!"

# Test single patient inference using exact API dataframe schema
test_patient_df = pd.DataFrame([{
    'age': 55.0,
    'hypertension': 1,
    'heart_disease': 0,
    'bmi': 31.2,
    'HbA1c_level': 6.8,
    'blood_glucose_level': 165.0,
    'gender_Male': 1,
    'gender_Other': 0,
    'smoking_history_current': 0,
    'smoking_history_ever': 0,
    'smoking_history_former': 1,
    'smoking_history_never': 0,
    'smoking_history_not current': 0
}], columns=loaded_features)

pred_val = int(loaded_estimator.predict(test_patient_df)[0])
prob_val = float(loaded_estimator.predict_proba(test_patient_df)[0][1])

print(f"Verification Inference Result:")
print(f"  Prediction:  {pred_val} ({'High Risk / Diabetic' if pred_val == 1 else 'Low Risk'})")
print(f"  Probability: {prob_val:.2%}")
print(f"  Status:      100% Verified against API schema!")


Verification Inference Result:
  Prediction:  1 (High Risk / Diabetic)
  Probability: 91.00%
  Status:      100% Verified against API schema!


---
## 9. Governance, Model Retention Rationale & Verification Instructions

### Production Retention Rationale
- The Random Forest model achieves high discriminative performance (**ROC-AUC: 0.9675**, **PR-AUC: 0.8513**), outperforming linear and dummy baselines.
- The model maintains balanced classification performance without catastrophic false alarm rates (Precision: 77.41%, Recall: 74.76% at 0.50 threshold).
- The Family Health Guardian backend does not rely on a naive hard threshold; it exposes continuous calibrated probability, using `prob >= 0.25` for moderate risk triage, yielding **86.26% clinical recall**.

### Instructions to Run this Notebook
1. **Local Environment**:
   ```bash
   cd backend
   python -m venv .venv
   .venv\Scripts\activate
   pip install -r requirements.txt
   jupyter notebook notebooks/welltree.ipynb
   ```
2. **Run Backend Regression Tests**:
   ```bash
   pytest tests/test_predictions.py -v
   ```
   All 16 ML service and prediction integration tests must pass.
